 # Stability Analysis



 The Jacobian matrix **J** at a cell state **x** characterises local dynamics:



 $$J_{ij}(x) = \frac{\partial \dot{x}_i}{\partial x_j} = W_{ij} \cdot \sigma'(x_j) - \gamma_i \delta_{ij}$$



 Key stability indicators:

 - **Trace(J)** < 0 → contracting (stabilising) dynamics

 - **Leading Re(λ)** < 0 → local attractor; > 0 → local repeller / saddle

 - **Rotational part** → oscillatory tendency



 Topics covered:

 1. Compute Jacobian matrices

 2. Jacobian summary statistics on UMAP

 3. Eigenvalue spectra per cluster

 4. Rotational dynamics

 5. Element-wise Jacobian analysis

 ## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch
import scHopfield as sch

DATA_PATH    = '/path/to/data/'
DATASET_FILE = 'hematopoiesis.h5ad'
MODEL_FILE   = 'model.h5sch'

CLUSTER_KEY  = 'cell_type'
SPLICED_KEY  = 'M_t'

CELL_TYPE_ORDER = ['Meg', 'Ery', 'MEP-like', 'HSC', 'GMP-like', 'Mon', 'Bas', 'Neu']

adata = sc.read_h5ad(DATA_PATH + DATASET_FILE)
sch.tl.load_model(adata, MODEL_FILE)
print(adata)

colors = {ct: f'C{i}' for i, ct in enumerate(CELL_TYPE_ORDER)}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')


 ## 4.1 Compute Jacobian Matrices



 **Warning**: This is memory-intensive for large datasets.

 Save immediately after computation.

In [ ]:
sch.tl.compute_jacobians(
    adata,
    spliced_key=SPLICED_KEY,
    cluster_key=CLUSTER_KEY,
    compute_eigenvectors=False,   # Set True only if eigenvectors are needed
    device=device
)

print("Jacobian eigenvalues stored in adata.obsm['jacobian_eigenvalues']")


In [ ]:
# Save to disk to avoid re-computation
# sch.tl.save_jacobians(
#     adata,
#     filename='jacobian_eigenvalues.h5',
#     cluster_key=CLUSTER_KEY,
#     compression='gzip'
# )
# print("Jacobians saved to disk.")


In [ ]:
# Load from disk later (if previously saved):
# sch.tl.load_jacobians(
#     adata,
#     filename='jacobian_eigenvalues.h5',
#     load_eigenvectors=False
# )
# sch.tl.compute_jacobian_stats(adata, store_in_obs=True)


 ## 4.2 Jacobian Statistics



 Extracts per-cell scalar summaries from eigenvalue arrays and stores them in

 `adata.obs`.

In [ ]:
sch.tl.compute_jacobian_stats(
    adata,
    filename=None,     # use adata.obsm if None, or pass a filename to load from disk
    store_in_obs=True
)

stats_cols = [
    'jacobian_eig1_real', 'jacobian_eig1_imag',
    'jacobian_positive_evals', 'jacobian_negative_evals',
    'jacobian_trace'
]
print("Jacobian statistics per cell:")
print(adata.obs[stats_cols].describe())


In [ ]:
# Visualise on UMAP
sc.pl.umap(
    adata,
    color=['jacobian_trace', 'jacobian_positive_evals', 'jacobian_eig1_real'],
    ncols=3,
    cmap='RdBu_r',
    vcenter=0
)


In [ ]:
# Summary per cluster
summary = adata.obs.groupby(CLUSTER_KEY).agg({
    'jacobian_positive_evals': ['mean', 'std'],
    'jacobian_negative_evals': ['mean', 'std'],
    'jacobian_trace':          ['mean', 'std'],
})
print("\nJacobian Statistics Summary:")
print(summary.round(3))


 ## 4.3 Eigenvalue Spectra

In [ ]:
# Eigenvalue spectrum in the complex plane (per cluster)
fig = sch.pl.plot_jacobian_eigenvalue_spectrum(
    adata,
    cluster_key=CLUSTER_KEY,
    order=CELL_TYPE_ORDER,
    colors=colors,
    figsize=(15, 15)
)
plt.show()


In [ ]:
# Boxplots of positive eigenvalue counts
fig = sch.pl.plot_jacobian_eigenvalue_boxplots(
    adata,
    cluster_key=CLUSTER_KEY,
    order=CELL_TYPE_ORDER,
    colors=colors,
    figsize=(15, 15)
)
plt.show()


In [ ]:
# Jacobian summary statistics as boxplots
fig = sch.pl.plot_jacobian_stats_boxplots(
    adata,
    cluster_key=CLUSTER_KEY,
    order=CELL_TYPE_ORDER,
    colors=colors,
    figsize=(15, 15)
)
plt.show()


In [ ]:
# Print extreme eigenvalues per cluster
print("Extreme eigenvalues per cluster:")
eigenvalues = adata.obsm['jacobian_eigenvalues']
for cluster in CELL_TYPE_ORDER:
    mask = (adata.obs[CLUSTER_KEY] == cluster).values
    evals_cluster = eigenvalues[mask]
    max_real = evals_cluster.real.max()
    min_real = evals_cluster.real.min()
    print(f"  {cluster:15s} | Max Re(λ): {max_real:8.3f} | Min Re(λ): {min_real:8.3f}")


 ## 4.4 Rotational Dynamics



 The antisymmetric part of J generates rotation in gene-expression phase space.

 High rotational magnitude → oscillatory / spiralling dynamics near the cell state.

In [ ]:
sch.tl.compute_rotational_part(
    adata,
    spliced_key=SPLICED_KEY,
    cluster_key=CLUSTER_KEY,
    device=device
)

print("Rotational magnitude stored in adata.obs['jacobian_rotational']")


In [ ]:
# Visualise rotational magnitude per cluster
sc.pl.umap(adata, color='jacobian_rotational', cmap='viridis')

# Boxplots
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5), tight_layout=True)
data = [adata.obs.loc[adata.obs[CLUSTER_KEY] == ct, 'jacobian_rotational'].values
        for ct in CELL_TYPE_ORDER]
ax.boxplot(data, labels=CELL_TYPE_ORDER, patch_artist=True,
           boxprops=dict(facecolor='lightblue'))
ax.set_xticklabels(CELL_TYPE_ORDER, rotation=30, ha='right')
ax.set_ylabel('Rotational magnitude')
ax.set_title('Jacobian rotational dynamics per cell type')
plt.show()


 ## 4.5 Element-wise Jacobian Analysis



 Compute specific partial derivatives ∂ẋᵢ/∂xⱼ for biologically motivated

 gene pairs, and visualise them on the UMAP.

In [ ]:
gene_pairs = [
    ('FLI1',  'KLF1'),  ('KLF1',  'FLI1'),
    ('FLI1',  'FLI1'),  ('KLF1',  'KLF1'),
    ('GATA1', 'GATA2'), ('GATA2', 'GATA1'),
    ('GATA1', 'KLF1'),  ('KLF1',  'GATA1'),
    ('GATA1', 'FLI1'),  ('FLI1',  'GATA1'),
    ('CEBPA', 'RUNX1'), ('RUNX1', 'CEBPA'),
    ('CEBPA', 'GATA2'), ('GATA2', 'CEBPA'),
    ('GATA2', 'RUNX1'), ('RUNX1', 'GATA2'),
    ('GATA2', 'GATA2'), ('RUNX1', 'RUNX1'),
]

sch.tl.compute_jacobian_elements(
    adata,
    gene_pairs=gene_pairs,
    spliced_key=SPLICED_KEY,
    cluster_key=CLUSTER_KEY,
    device=device,
    store_in_obs=True
)

print("Jacobian elements stored in adata.obs with names like: jacobian_df_GATA1_dx_GATA2")


In [ ]:
# FLI1–KLF1 mutual regulation
fig = sch.pl.plot_jacobian_element_grid(
    adata,
    gene_pairs=[('FLI1', 'KLF1'), ('KLF1', 'FLI1'),
                ('FLI1', 'FLI1'), ('KLF1', 'KLF1')],
    ncols=2,
    figsize=(10, 10),
    s=10,
    alpha=0.7
)
plt.suptitle('FLI1–KLF1 Jacobian Elements', fontsize=16, y=1.02)
plt.show()


In [ ]:
# GATA1 regulon
fig = sch.pl.plot_jacobian_element_grid(
    adata,
    gene_pairs=[('GATA1', 'GATA2'), ('GATA2', 'GATA1'),
                ('GATA1', 'KLF1'),  ('KLF1',  'GATA1'),
                ('GATA1', 'FLI1'),  ('FLI1',  'GATA1')],
    ncols=2,
    figsize=(10, 15),
    s=10,
    alpha=0.7
)
plt.suptitle('GATA1 Jacobian Elements', fontsize=16, y=1.02)
plt.show()


In [ ]:
# CEBPA–RUNX1–GATA2 cross-regulation
fig = sch.pl.plot_jacobian_element_grid(
    adata,
    gene_pairs=[('CEBPA', 'RUNX1'), ('RUNX1', 'CEBPA'),
                ('GATA2', 'RUNX1'), ('RUNX1', 'GATA2'),
                ('GATA2', 'GATA2'), ('RUNX1', 'RUNX1')],
    ncols=2,
    figsize=(10, 15),
    s=10,
    alpha=0.7
)
plt.suptitle('CEBPA–RUNX1–GATA2 Jacobian Elements', fontsize=16, y=1.02)
plt.show()
